# vast.ai — Notebook 1: SET UP the box (clone + deps + 274 games)

One-time setup for the RTX PRO 6000. Run top-to-bottom, then open **Notebook 2**.
The 252 community games are .gitignored, so this re-clones them. Set GIT_TOKEN if
you'll push commits from the box.

In [ ]:
# Cell 1 — clone (or update) the repo
GIT_TOKEN = ""          # PAT, only needed to PUSH commits from the box
import os, subprocess
auth = (GIT_TOKEN + "@") if GIT_TOKEN else ""
url  = f"https://{auth}github.com/shreyasmahimkar/arc-agi-3.git"
if os.path.exists("/workspace/arc3/.git"):
    print(subprocess.run(["git","-C","/workspace/arc3","pull"],capture_output=True,text=True).stdout)
else:
    print(subprocess.run(["git","clone","--branch","main",url,"/workspace/arc3"],capture_output=True,text=True).stderr)
print("repo present:", os.path.exists("/workspace/arc3/.git"))

In [ ]:
# Cell 2 — use the kernel's torch (vast jupyter already runs in a torch venv); add arcengine to it
import sys, subprocess, shutil
try:
    import torch; PY = sys.executable
except ImportError:
    def has_torch(p):
        try: return bool(p) and subprocess.run([p,"-c","import torch"],capture_output=True).returncode==0
        except FileNotFoundError: return False
    PY = next((p for p in [shutil.which("python"),shutil.which("python3"),"/venv/main/bin/python","/opt/conda/bin/python"] if has_torch(p)), None)
assert PY, "no torch python found"
W = "/workspace/arc3/arc-prize-2026-arc-agi-3/arc_agi_3_wheels"
!{PY} -m pip install -q --ignore-installed blinker
!{PY} -m pip install -q --find-links {W} {W}/arcengine-0.9.3-py3-none-any.whl {W}/arc_agi-0.9.8-py3-none-any.whl
open("/workspace/PY.txt","w").write(PY)
!{PY} -c "import torch,arcengine; print('OK',torch.__version__,torch.cuda.is_available(),torch.cuda.get_device_name(0))"
print("PY =", PY, "(saved to /workspace/PY.txt)")

In [ ]:
# Cell 3 — bring in the 252 community games (gitignored)
ENV = "/workspace/arc3/arc-prize-2026-arc-agi-3/environment_files"
!rm -rf /tmp/arc-interactive && git clone --depth 1 https://github.com/theredbluepill/arc-interactive /tmp/arc-interactive
!cp -rn /tmp/arc-interactive/environment_files/* {ENV}/
import os; print("games now:", len([d for d in os.listdir(ENV) if os.path.isdir(os.path.join(ENV,d))]), "(expect ~274)")

In [ ]:
# Cell 4 — git identity for committing from the box
!git -C /workspace/arc3 config user.email "shreyasmahimkar@gmail.com"
!git -C /workspace/arc3 config user.name  "shreyasmahimkar"
print("git configured")

In [ ]:
# Cell 5 — sanity: discovery + a quick BFS solve
PY = open("/workspace/PY.txt").read().strip(); V19 = "/workspace/arc3/CommunitySolutions/chronos_solver/v19"
!cd {V19} && {PY} -c "import solve_all; print('discovered games:', len(solve_all.ALL_GAMES))"
!cd {V19} && {PY} solve_all.py --games ls20 --bfs-timeout 30 --max-levels 2 2>&1 | grep -E "SOLVED|campaign" | head
print("SETUP COMPLETE -> open Notebook 2")